# FairDerm: Fair Skin Lesion Classification

This notebook trains and evaluates skin lesion classifiers with a focus on fairness across skin tones.

**Methods compared:**
1. Baseline (CE + Uniform sampling)
2. Mixup (CE + Uniform + Mixup augmentation)
3. Reweighted (Weighted CE + Class-weighted sampling)
4. Focal Loss (Focal Loss gamma=2)
5. Proposed (CE + Skin-tone-adaptive sampling + Mixup)

**Dataset:** Fitzpatrick17k (training/validation)

**Model:** EfficientNet-B2

## 1. Setup

In [ ]:
# Mount Google Drive (for Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install timm --quiet

In [ ]:
# Add fairderm package to path (if running from Colab)
import sys
sys.path.insert(0, '/content/drive/MyDrive/thesis/code')

# Import standard libraries
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Import custom fairderm modules
from fairderm import (
    SkinLesionDataset,
    create_model,
    SkinToneAdaptiveSampler,
    get_class_weights,
    get_train_transforms,
    get_eval_transforms,
    run_experiment,
    save_experiment_results,
)
from fairderm.configs import (
    get_baseline_config,
    get_mixup_config,
    get_reweighted_config,
    get_focal_config,
    get_proposed_config,
)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Configuration

In [ ]:
import os

# Paths - update these for your setup
PROJECT_DIR = '/content/drive/MyDrive/thesis'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')

CSV_PATH = os.path.join(DATA_DIR, 'fitzpatrick17k_processed.csv')

os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")

## 3. Load and Prepare Data

In [ ]:
# Load the processed CSV
df = pd.read_csv(CSV_PATH)

# Show dataset summary
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"\nTotal samples: {len(df)}")
print("\nLabel distribution:")
print(df['label_num'].value_counts())
print("\nSkin tone group distribution:")
print(df['tone_group'].value_counts())

# Cross-tabulation to see the breakdown
print("\nLabel x Skin Tone Group:")
print(pd.crosstab(df['label'], df['tone_group']))

In [ ]:
# Create stratified train/val split
# Stratify by BOTH label and skin tone to ensure balanced representation
df['stratify_key'] = df['label'].astype(str) + '_' + df['tone_group']

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['stratify_key']
)

train_df = train_df.drop('stratify_key', axis=1).reset_index(drop=True)
val_df = val_df.drop('stratify_key', axis=1).reset_index(drop=True)

print(f"Train set: {len(train_df)} samples")
print(f"Val set: {len(val_df)} samples")

print("\nTraining skin tone distribution:")
print(train_df['tone_group'].value_counts())

print("\nValidation skin tone distribution:")
print(val_df['tone_group'].value_counts())

In [ ]:
# Create datasets with transforms
# Resize to 260, then crop to 224 (random for train, center for eval)
train_transform = get_train_transforms()
eval_transform = get_eval_transforms()

train_dataset = SkinLesionDataset(train_df, transform=train_transform, return_group=True)
val_dataset = SkinLesionDataset(val_df, transform=eval_transform, return_group=True)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

## 4. Run Experiments

We run each method and compare results.

In [ ]:
# Compute class weights for reweighted baseline
class_weights = get_class_weights(train_df, label_col='label_num')
print(f"Class weights: {class_weights}")

In [ ]:
# Store all results
all_results = {}

### 4.1 Baseline

In [ ]:
config = get_baseline_config()
results = run_experiment(train_dataset, val_dataset, config, seed=42, device=device)
all_results['baseline_seed42'] = results

# Save model
save_experiment_results(results, 'baseline_seed42', RESULTS_DIR)

### 4.2 Mixup

In [ ]:
config = get_mixup_config()
results = run_experiment(train_dataset, val_dataset, config, seed=42, device=device)
all_results['mixup_seed42'] = results

save_experiment_results(results, 'mixup_seed42', RESULTS_DIR)

### 4.3 Reweighted

In [ ]:
config = get_reweighted_config(class_weights, device=device)
results = run_experiment(train_dataset, val_dataset, config, seed=42, device=device)
all_results['reweighted_seed42'] = results

save_experiment_results(results, 'reweighted_seed42', RESULTS_DIR)

### 4.4 Focal Loss

In [ ]:
config = get_focal_config()
results = run_experiment(train_dataset, val_dataset, config, seed=42, device=device)
all_results['focalloss_seed42'] = results

save_experiment_results(results, 'focalloss_seed42', RESULTS_DIR)

### 4.5 Proposed

In [ ]:
config = get_proposed_config()
results = run_experiment(train_dataset, val_dataset, config, seed=42, device=device)
all_results['proposed_seed42'] = results

save_experiment_results(results, 'proposed_seed42', RESULTS_DIR)

# Plot adaptive sampler history
if results['adaptive_sampler']:
    results['adaptive_sampler'].plot_history(
        save_path=os.path.join(RESULTS_DIR, 'proposed_seed42', 'sampler_history.png')
    )

## 5. Training Complete

All models have been trained and saved to the results directory. Each model folder contains:
- `model.pt` - Trained model weights
- `metrics.json` - Final validation metrics
- `training_history.csv` - Per-epoch training metrics
- `training_history.png` - Loss/accuracy curves

**For results analysis and visualization, see:** `fairderm_results_analysis.ipynb`

In [ ]:
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"\nModels saved to: {RESULTS_DIR}")
print(f"\nTrained models:")
for name in all_results.keys():
    print(f"  - {name}")
print("\nRun 'fairderm_results_analysis.ipynb' for detailed analysis and visualizations.")